In [0]:
df=spark.read.format("csv").option("header",True).option("inferSchema",True).load("/Volumes/dbtsparkproject/source/source_data/customers")

In [0]:
display(df)

In [0]:
schema_customers=df.schema
schema_customers


In [0]:
entities=['customers','trips','payments','drivers','vehicles','locations']

In [0]:
for entity in entities:
    # 1. Added 'f' prefix here
    df_batch = (
        spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(f"/Volumes/dbtsparkproject/source/source_data/{entity}")
    )

    schema_entity = df_batch.schema

    df = (
        spark.readStream.format("csv")
        .option("header", True)
        .schema(schema_entity)
        .load(f"/Volumes/dbtsparkproject/source/source_data/{entity}")
    )

    # 2. Fixed f-string on checkpointLocation from /entity to /{entity}
    query = (
        df.writeStream.format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            f"/Volumes/dbtsparkproject/bronze/checkpoint_volume/{entity}",
        )
        .trigger(once=True)
        .toTable(f"dbtsparkproject.bronze.{entity}")
    )

    